# Watch a run, epoch by epoch

Runs **one ladder cell** and streams every epoch into the notebook as it happens.

This is deliberately a *subprocess*, not an in-process call: `build_command` produces
byte for byte the argv that `pool.py` would run, so what you watch here is exactly
what the sweep does. Nothing is special-cased for the notebook.

**Run order matters.** Every sparse rung starts from the dense checkpoint *of the same
seed*, so `A0` has to exist at your chosen seed before anything else will schedule.

## VS Code setup

1. Install the **Remote - SSH** and **Jupyter** extensions.
2. `Ctrl+Shift+P` -> *Remote-SSH: Add New SSH Host* -> paste the SSH command from the
   Vast.ai instance card (the `ssh -p <port> root@<host>` one), then *Connect*.
3. *File -> Open Folder* -> `/workspace/BaCP`.
4. Open this notebook, then **Select Kernel -> Python Environments -> `/venv/main/bin/python`**.
   That interpreter is the one with the CUDA build of torch. The system `python3` does
   not have torch at all, which is what breaks a run started inside tmux.

In [ ]:
import os, sys, re, json, time, subprocess
from pathlib import Path

# The repo on the box. Override with BACP_REPO if you open this elsewhere.
REPO = Path(os.environ.get('BACP_REPO', '/workspace/BaCP'))

# Results go on /workspace so they survive a pod stop (they do NOT survive a
# terminate). Checkpoints are separate -- training_utils puts those under
# project/scripts/research/, relative to the script's cwd.
os.environ.setdefault('BACP_RESULTS_DIR', '/workspace/bacp_results')

# Without this the child buffers stdout and every epoch arrives in one lump at
# the end instead of live. This single line is what makes the cell below work.
os.environ['PYTHONUNBUFFERED'] = '1'

PY = sys.executable
for p in ('project', 'project/experiments', 'project/scripts'):
    q = str(REPO / p)
    if q not in sys.path:
        sys.path.insert(0, q)

import torch
import manifest as M
import runner as R

print(f'repo      {REPO}')
print(f'results   {os.environ["BACP_RESULTS_DIR"]}')
print(f'python    {PY}')
print(f'torch     {torch.__version__}  cuda {torch.version.cuda}')
if torch.cuda.is_available():
    cap = ''.join(map(str, torch.cuda.get_device_capability(0)))
    print(f'gpus      {torch.cuda.device_count()} x {torch.cuda.get_device_name(0)}  sm_{cap}')
else:
    print('gpus      !! NONE VISIBLE -- do not start a run')
n = len(list((Path(os.environ['BACP_RESULTS_DIR']) / 'runs').glob('*.json'))) \
    if (Path(os.environ['BACP_RESULTS_DIR']) / 'runs').exists() else 0
print(f'records   {n}')

## 1. Pick a cell

`EPOCHS` and `SPARSITY` override the manifest for a quick probe. Leave them `None`
for the real protocol (250 epochs, 0.999). Anything you override here makes the run
a **diagnostic, not a ladder measurement** — the record still gets written, so give it
a distinct `GROUP_SUFFIX` to keep it out of the ladder's key space.

In [ ]:
RUNG          = 'A3'    # A0 dense | A1..A6 substrate | C-null C0 C0b C1 C2 C3 | D1 D2
SEED          = 1
TIER          = 1
GPU           = 0

EPOCHS        = None    # None = protocol default (250). 80 catches the known divergence.
SPARSITY      = None    # None = manifest default (0.999). 0.99 has published comparators.
LEARNING_RATE = None    # None = 0.1. Lower this to test the divergence hypothesis.
GROUP_SUFFIX  = ''      # e.g. '.probe' -- set whenever you override anything above.

cell = [c for c in M.cells(TIER, rungs=[RUNG]) if c['seed'] == SEED][0]
cell['config']['num_workers'] = 8
if EPOCHS:                    cell['config']['epochs'] = EPOCHS
if LEARNING_RATE:             cell['config']['learning_rate'] = LEARNING_RATE
if SPARSITY is not None and 'target_sparsity' in cell['config']:
    cell['config']['target_sparsity'] = SPARSITY
if GROUP_SUFFIX:              cell['key'] = cell['key'] + GROUP_SUFFIX

# Sparse rungs resolve their starting weights from a dense record of the SAME
# seed. This raises if A0 has not run at this seed -- which is correct: the
# alternative is silently pairing five 'independent' seeds to one checkpoint.
try:
    cell = R.attach_checkpoint(cell)
    print(f'checkpoint  {cell["config"].get("trained_weights")}')
except Exception as exc:
    print(f'checkpoint  NOT RESOLVED -- {type(exc).__name__}: {exc}')
    print('            expected for A0; every other rung needs A0 at this seed first.')

print(f'\nkey       {cell["key"]}')
print(f'script    {cell["script"]}')
print(f'changes   {cell["changes"]}')
print('config:')
for k, v in sorted(cell['config'].items()):
    print(f'  {k:24s} {v}')

## 2. Run it, streaming every epoch

Prints one line per epoch with a running s/epoch and an ETA, and shouts the moment
the loss becomes NaN. Interrupt the kernel to stop — the child is terminated with it.

In [ ]:
EPOCH_RE = re.compile(r'Epoch \[(\d+)/(\d+)\]')

def _field(name, line):
    """Pull `name: value` out of a log line. Returns nan for a literal 'nan'."""
    m = re.search(rf'{name}: (nan|-?[0-9.]+(?:[eE][-+]?\d+)?)', line)
    if not m:
        return None
    try:
        return float(m.group(1))
    except ValueError:
        return float('nan')

def watch(cell, gpu=0, echo_other=True):
    argv = R.build_command(cell, python=PY)
    env = {**os.environ,
           'CUDA_VISIBLE_DEVICES': str(gpu),
           'BACP_EXPERIMENT_GROUP': cell['key']}
    hist = {'epoch': [], 'acc': [], 'loss': [], 'sparsity': [], 'val_loss': []}
    first_nan, t0 = None, time.time()

    print(' '.join(argv[1:]), '\n')
    proc = subprocess.Popen(argv, cwd=str(REPO / 'project' / 'scripts'),
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env=env)
    try:
        for raw in proc.stdout:
            line = raw.rstrip()
            m = EPOCH_RE.search(line)
            if not m:
                if echo_other and line.strip():
                    print(line)
                continue
            ep, tot = int(m.group(1)), int(m.group(2))
            acc = _field('accuracy', line)
            loss = _field('loss', line)
            sp = _field('sparsity', line)
            hist['epoch'].append(ep)
            hist['acc'].append(acc)
            hist['loss'].append(loss)
            hist['sparsity'].append(sp)
            hist['val_loss'].append(_field('val_loss', line))

            if loss is not None and loss != loss and first_nan is None:
                first_nan = ep
                print(f'\n  *** LOSS WENT NaN AT EPOCH {ep}/{tot} ***\n')

            per = (time.time() - t0) / max(ep, 1)
            print(f'epoch {ep:4d}/{tot}  acc {acc if acc is None else f"{acc:7.4f}"}'
                  f'  loss {loss}  sparsity {sp}'
                  f'  [{per:5.1f}s/ep  eta {per*(tot-ep)/60:5.1f}m]')
    except KeyboardInterrupt:
        proc.terminate()
        print('\ninterrupted; child terminated')
    finally:
        proc.wait()

    mins = (time.time() - t0) / 60
    print(f'\nexit {proc.returncode} after {mins:.1f} min'
          + (f'   FIRST NaN AT EPOCH {first_nan}' if first_nan else ''))
    return hist, first_nan

hist, first_nan = watch(cell, gpu=GPU)

## 3. Plot what just happened

Run any time after (or during, from a second kernel) — `hist` is a plain dict.

In [ ]:
import matplotlib.pyplot as plt

def plot(hist, first_nan=None, title=''):
    fig, ax = plt.subplots(1, 3, figsize=(14, 3.4))
    ax[0].plot(hist['epoch'], hist['acc'], lw=1)
    ax[0].set_title('val accuracy (%)')
    ax[0].axhline(10.0, color='grey', ls=':', lw=1)   # chance on CIFAR-10
    # NaN cannot be drawn, so plot only the finite prefix and let the dashed
    # line mark where it stopped being finite.
    fin = [(e, l) for e, l in zip(hist['epoch'], hist['loss'])
           if l is not None and l == l]
    ax[1].plot([e for e, _ in fin], [l for _, l in fin], lw=1)
    ax[1].set_title('train loss (finite only)')
    ax[2].plot(hist['epoch'], hist['sparsity'], lw=1)
    ax[2].set_title('sparsity')
    for a in ax:
        a.set_xlabel('epoch')
        if first_nan:
            a.axvline(first_nan, color='crimson', ls='--', lw=1)
    fig.suptitle(title or cell['key'], fontsize=9)
    plt.tight_layout()
    plt.show()

plot(hist, first_nan)

## 4. Probe several configs back to back

The stability question is *which pruners diverge*, and we have only ever tested RigL.
This runs each family for a truncated number of epochs — long enough to pass the
known divergence points (C-null went NaN at ~10, C0 at ~64) — and reports where each
one broke, if it did.

At `0.99` there are published comparators for this exact cell: RigL 92.92 ± 0.18,
EAST 93.51 ± 0.13, SET 93.09 ± 0.15 (EAST v4 Table 1). That makes 99% the level
where a parity check actually has teeth.

In [ ]:
PROBE_RUNGS    = ['A2', 'A3', 'A5', 'A6']   # uniform mag, ERK mag, RigL, EAST
PROBE_EPOCHS   = 80
PROBE_SPARSITY = 0.99
PROBE_SEED     = 1

probe = {}
for rung in PROBE_RUNGS:
    c = [x for x in M.cells(TIER, rungs=[rung]) if x['seed'] == PROBE_SEED][0]
    c['config']['num_workers'] = 8
    c['config']['epochs'] = PROBE_EPOCHS
    if 'target_sparsity' in c['config']:
        c['config']['target_sparsity'] = PROBE_SPARSITY
    c['key'] = f'{c["key"]}.probe{PROBE_SPARSITY}'
    try:
        c = R.attach_checkpoint(c)
    except Exception as exc:
        print(f'{rung}: NOT SCHEDULED -- {exc}')
        continue
    print(f'\n{"="*70}\n{rung}  ({c["changes"]})\n{"="*70}')
    h, nan_at = watch(c, gpu=GPU, echo_other=False)
    best = max([a for a in h['acc'] if a is not None] or [float("nan")])
    probe[rung] = {'hist': h, 'first_nan': nan_at, 'best_acc': best}

print(f'\n{"rung":6s} {"best acc":>9s} {"first NaN":>10s}')
for rung, r in probe.items():
    print(f'{rung:6s} {r["best_acc"]:9.2f} {str(r["first_nan"] or "-"):>10s}')

## 5. The ladder table and the gates

Reads every record written so far. **Pass the tier explicitly** — `report()` otherwise
falls back to `BACP_TIER`, which defaults to 0, and tier 0 uses `smoke.*` keys that
match none of these records, so you get an empty table with no error.

In [ ]:
import importlib
import ladder
importlib.reload(ladder)

out = ladder.report(TIER)